In [65]:
import pandas as pd
import numpy as np

games = pd.read_csv('../data/clean/clean_game.csv')

# Ensure date column is datetime
games['game_date'] = pd.to_datetime(games['game_date'], errors='coerce')

games.head()

,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type,home_win
0,21946,1610610035,HUS,Toronto Huskies,24600001,1946-11-01,HUS vs. NYK,L,0,25.0,...,NaN,NaN,NaN,NaN,NaN,68.0,2,0,Regular Season,0
1,21946,1610610034,BOM,St. Louis Bombers,24600003,1946-11-02,BOM vs. PIT,W,0,20.0,...,NaN,NaN,NaN,NaN,25.0,51.0,-5,0,Regular Season,1
2,21946,1610610032,PRO,Providence Steamrollers,24600002,1946-11-02,PRO vs. BOS,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,53.0,-6,0,Regular Season,1
3,21946,1610610025,CHS,Chicago Stags,24600004,1946-11-02,CHS vs. NYK,W,0,21.0,...,NaN,NaN,NaN,NaN,22.0,47.0,-16,0,Regular Season,1
4,21946,1610610028,DEF,Detroit Falcons,24600005,1946-11-02,DEF vs. WAS,L,0,10.0,...,NaN,NaN,NaN,NaN,NaN,50.0,17,0,Regular Season,0


In [66]:
home_cols = [c for c in games.columns if c.endswith('_home')]
away_cols = [c for c in games.columns if c.endswith('_away')]

In [67]:
home_df = games[['game_id','season_id','game_date'] + home_cols].copy()
home_df['team_id'] = games['team_id_home']
home_df['team_name'] = games['team_name_home']
home_df['team_win'] = games['home_win']
home_df['home_away_flag'] = 'home'

In [68]:
away_df = games[['game_id','season_id','game_date'] + away_cols].copy()
away_df['team_id'] = games['team_id_away']
away_df['team_name'] = games['team_name_away']
away_df['team_win'] = 1 - games['home_win']
away_df['home_away_flag'] = 'away'

In [69]:
home_df = home_df.rename(columns={c: "stat_" + c.replace('_home', '') for c in home_cols})
away_df = away_df.rename(columns={c: "stat_" + c.replace('_away', '') for c in away_cols})

In [70]:
common_cols = sorted(set(home_df.columns).intersection(set(away_df.columns)))
home_df = home_df[common_cols]
away_df = away_df[common_cols]

In [71]:
team_games = pd.concat([home_df, away_df], ignore_index=True)
team_games = team_games.sort_values(['team_id', 'game_date']).reset_index(drop=True)
team_games.head()

,game_date,game_id,home_away_flag,season_id,stat_ast,stat_blk,stat_dreb,stat_fg3_pct,stat_fg3a,stat_fg3m,...,stat_stl,stat_team_abbreviation,stat_team_id,stat_team_name,stat_tov,stat_video_available,stat_wl,team_id,team_name,team_win
0,2014-10-05,11400003,away,12014,19.0,7.0,28.0,0.258,31.0,8.0,...,7.0,MTA,41,Tel Aviv Maccabi Electra,9.0,0,L,41,Tel Aviv Maccabi Electra,0
1,2014-10-07,11400011,away,12014,15.0,5.0,32.0,0.231,26.0,6.0,...,4.0,MTA,41,Tel Aviv Maccabi Electra,21.0,0,L,41,Tel Aviv Maccabi Electra,0
2,2007-10-18,10700067,home,12007,15.0,4.0,22.0,0.419,31.0,13.0,...,2.0,CHN,45,China Team China,17.0,0,L,45,China Team China,0
3,2010-10-03,11000002,away,12010,14.0,2.0,25.0,0.316,19.0,6.0,...,10.0,MAC,93,Haifa Maccabi Haifa,21.0,0,L,93,Haifa Maccabi Haifa,0
4,2012-10-11,11200029,away,12012,23.0,5.0,26.0,0.381,21.0,8.0,...,9.0,MAC,93,Haifa Maccabi Haifa,14.0,0,L,93,Haifa Maccabi Haifa,0


In [72]:
stat_cols = [
    c for c in team_games.columns
    if c.startswith('stat_') and pd.api.types.is_numeric_dtype(team_games[c])
]

In [73]:
windows = [3, 5, 10]

for stat in stat_cols:
    for w in windows:
        team_games[f'{stat}_roll_{w}'] = (
            team_games
            .groupby('team_id')[stat]
            .rolling(w)
            .mean()
            .shift(1)
            .reset_index(level=0, drop=True)
        )

In [74]:
streak_windows = [1,3,5,10]

for w in streak_windows:
    team_games[f'win_streak_{w}'] = (
        team_games
        .groupby('team_id')['team_win']
        .rolling(w)
        .sum()
        .shift(1)
        .reset_index(level=0, drop=True)
    )

In [75]:
team_games['prev_game_date'] = team_games.groupby('team_id')['game_date'].shift(1)
team_games['rest_days'] = (team_games['game_date'] - team_games['prev_game_date']).dt.days
team_games['is_b2b'] = (team_games['rest_days'] <= 1).astype(int)
team_games['rest_3plus'] = (team_games['rest_days'] >= 3).astype(int)

In [76]:
team_games = team_games.sort_values(['team_id','season_id','game_date'])

team_games['season_win_pct'] = (
    team_games
    .groupby(['team_id','season_id'])['team_win']
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=[0,1], drop=True)
)

In [77]:
team_games['season_wins'] = (
    team_games
    .groupby(['team_id','season_id'])['team_win']
    .cumsum()
    .shift(1)
)

team_games['season_game_number'] = (
    team_games.groupby(['team_id','season_id']).cumcount()
)

In [78]:
home_feats = team_games[team_games['home_away_flag']=='home'].copy()
away_feats = team_games[team_games['home_away_flag']=='away'].copy()

In [79]:
home_feats = home_feats.add_prefix('home_')
away_feats = away_feats.add_prefix('away_')

In [80]:
model_data = games[['game_id','home_win','team_id_home','team_id_away']].copy()

model_data = model_data.merge(
    home_feats, left_on='game_id', right_on='home_game_id', how='left'
).merge(
    away_feats, left_on='game_id', right_on='away_game_id', how='left'
)

In [81]:
model_data = model_data.drop(columns=[
    'home_game_id','away_game_id',
    'team_id_home','team_id_away',
    'home_team_id','away_team_id'
], errors='ignore')

In [82]:
numeric_home_stats = [
    c for c in model_data.columns
    if c.startswith('home_stat_') and pd.api.types.is_numeric_dtype(model_data[c])
]

numeric_away_stats = [
    c for c in model_data.columns
    if c.startswith('away_stat_') and pd.api.types.is_numeric_dtype(model_data[c])
]

In [83]:

diff_cols = []

# Identify numeric-only home/away stat columns
numeric_home_stats = [
    c for c in model_data.columns
    if c.startswith('home_stat_') and pd.api.types.is_numeric_dtype(model_data[c])
]

numeric_away_stats = [
    c for c in model_data.columns
    if c.startswith('away_stat_') and pd.api.types.is_numeric_dtype(model_data[c])
]

# Compute differences safely
for home_col in numeric_home_stats:
    base = home_col.replace('home_stat_', '')
    away_col = 'away_stat_' + base

    if away_col in numeric_away_stats:
        diff_col = 'diff_' + base

        model_data[diff_col] = (
            pd.to_numeric(model_data[home_col], errors='coerce')
            - pd.to_numeric(model_data[away_col], errors='coerce')
        )
        
        diff_cols.append(diff_col)

# Fill NaNs created from coercion or missing values
model_data = model_data.fillna(0)

# Quick check
print("Created diff columns:", len(diff_cols))
print(diff_cols[:15])

Created diff columns: 84
['diff_ast', 'diff_blk', 'diff_dreb', 'diff_fg3_pct', 'diff_fg3a', 'diff_fg3m', 'diff_fg_pct', 'diff_fga', 'diff_fgm', 'diff_ft_pct', 'diff_fta', 'diff_ftm', 'diff_oreb', 'diff_pf', 'diff_plus_minus']


In [84]:
model_data = model_data.fillna(0)

In [85]:
cols_to_drop = [
    c for c in model_data.columns
    if 'team_id' in c or 'team_name' in c or 'matchup' in c
]

model_data = model_data.drop(columns=cols_to_drop, errors='ignore')

In [86]:
cols_to_drop = [
    c for c in model_data.columns if 'team_id' in c and 'roll' in c
]

model_data = model_data.drop(columns=cols_to_drop, errors='ignore')

In [87]:
model_data.to_csv('../data/modelling/model_data.csv', index=False)

model_data.head(), model_data.shape

(    game_id  home_win home_game_date home_home_away_flag  home_season_id  \
 0  24600001         0     1946-11-01                home           21946   
 1  24600003         1     1946-11-02                home           21946   
 2  24600002         1     1946-11-02                home           21946   
 3  24600004         1     1946-11-02                home           21946   
 4  24600005         0     1946-11-02                home           21946   
 
    home_stat_ast  home_stat_blk  home_stat_dreb  home_stat_fg3_pct  \
 0            0.0            0.0             0.0                0.0   
 1            0.0            0.0             0.0                0.0   
 2            0.0            0.0             0.0                0.0   
 3            0.0            0.0             0.0                0.0   
 4            0.0            0.0             0.0                0.0   
 
    home_stat_fg3a  ...  diff_reb_roll_10  diff_stl_roll_3  diff_stl_roll_5  \
 0             0.0  ...      